In [41]:
# ==========================================
# Hybrid Recommendation System
# ==========================================

import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings("ignore")

In [42]:
# ==========================================
# Hybrid Recommendation Weights
# ==========================================

# Weight assigned to content similarity
SIMILARITY_WEIGHT = 0.70

# Weight assigned to movie rating
RATING_WEIGHT = 0.20

# Weight assigned to popularity
POPULARITY_WEIGHT = 0.10

In [43]:
# Load processed movie dataset
movies = pickle.load(open("../models/movies.pkl", "rb"))

# Load similarity matrix
similarity = pickle.load(open("../models/similarity.pkl", "rb"))

movies.head()

,Title,Tags,Genre,Overview,Original_Language,Popularity,Vote_Average,Vote_Count,Poster_Url,Year,Popularity_Score,Rating_Score
0,Spider-Man: No Way Home,action adventur scienc fiction peter parker un...,"action, adventure, science fiction",peter parker is unmasked and no longer able to...,en,5083.954,8.3,8940.0,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,2021,1.000000,0.83
1,The Batman,crime mysteri thriller second year fight crime...,"crime, mystery, thriller","in his second year of fighting crime, batman u...",en,3827.658,8.1,1151.0,https://image.tmdb.org/t/p/original/74xTEgt7R3...,2022,0.752239,0.81
2,No Exit,thriller strand rest stop mountain blizzard re...,thriller,stranded at a rest stop in the mountains durin...,en,2618.087,6.3,122.0,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,2022,0.513693,0.63
3,Encanto,anim comedi famili fantasi tale extraordinari ...,"animation, comedy, family, fantasy","the tale of an extraordinary family, the madri...",en,2402.201,7.7,5076.0,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,2021,0.471117,0.77
4,The King's Man,action adventur thriller war collect histori w...,"action, adventure, thriller, war",as a collection of history's worst tyrants and...,en,1895.511,7.0,1793.0,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,2021,0.371190,0.70


In [44]:
print(movies.shape)



(9826, 12)


In [45]:
movies.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9826 entries, 0 to 9825
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Title              9826 non-null   object 
 1   Tags               9826 non-null   object 
 2   Genre              9826 non-null   object 
 3   Overview           9826 non-null   object 
 4   Original_Language  9826 non-null   object 
 5   Popularity         9826 non-null   float64
 6   Vote_Average       9826 non-null   float64
 7   Vote_Count         9826 non-null   float64
 8   Poster_Url         9826 non-null   object 
 9   Year               9826 non-null   int64  
 10  Popularity_Score   9826 non-null   float64
 11  Rating_Score       9826 non-null   float64
dtypes: float64(5), int64(1), object(6)
memory usage: 921.3+ KB


In [46]:

movies.head()

,Title,Tags,Genre,Overview,Original_Language,Popularity,Vote_Average,Vote_Count,Poster_Url,Year,Popularity_Score,Rating_Score
0,Spider-Man: No Way Home,action adventur scienc fiction peter parker un...,"action, adventure, science fiction",peter parker is unmasked and no longer able to...,en,5083.954,8.3,8940.0,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,2021,1.000000,0.83
1,The Batman,crime mysteri thriller second year fight crime...,"crime, mystery, thriller","in his second year of fighting crime, batman u...",en,3827.658,8.1,1151.0,https://image.tmdb.org/t/p/original/74xTEgt7R3...,2022,0.752239,0.81
2,No Exit,thriller strand rest stop mountain blizzard re...,thriller,stranded at a rest stop in the mountains durin...,en,2618.087,6.3,122.0,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,2022,0.513693,0.63
3,Encanto,anim comedi famili fantasi tale extraordinari ...,"animation, comedy, family, fantasy","the tale of an extraordinary family, the madri...",en,2402.201,7.7,5076.0,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,2021,0.471117,0.77
4,The King's Man,action adventur thriller war collect histori w...,"action, adventure, thriller, war",as a collection of history's worst tyrants and...,en,1895.511,7.0,1793.0,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,2021,0.371190,0.70


In [47]:
scaler = MinMaxScaler()

In [48]:
movies["Popularity_Score"] = scaler.fit_transform(
    movies[["Popularity"]]
)

In [49]:
movies["Rating_Score"] = scaler.fit_transform(
    movies[["Vote_Average"]]
)

In [50]:
movies[
    [
        "Popularity",
        "Popularity_Score",
        "Vote_Average",
        "Rating_Score"
    ]
].head()

,Popularity,Popularity_Score,Vote_Average,Rating_Score
0,5083.954,1.000000,8.3,0.83
1,3827.658,0.752239,8.1,0.81
2,2618.087,0.513693,6.3,0.63
3,2402.201,0.471117,7.7,0.77
4,1895.511,0.371190,7.0,0.70


In [51]:
# ==========================================
# Validate Weights
# ==========================================

TOTAL_WEIGHT = (
    SIMILARITY_WEIGHT +
    RATING_WEIGHT +
    POPULARITY_WEIGHT
)

if abs(TOTAL_WEIGHT - 1.0) > 0.0001:
    raise ValueError(
        f"Weights must sum to 1.0 (Current = {TOTAL_WEIGHT})"
    )

print("✅ Hybrid weights validated successfully.")

✅ Hybrid weights validated successfully.


In [52]:
# ==========================================
# Calculate Hybrid Score
# ==========================================

def calculate_hybrid_score(similarity_score,
                           rating_score,
                           popularity_score):
    """
    Calculates the final hybrid recommendation score.

    Parameters
    ----------
    similarity_score : float
        Cosine similarity score (0-1)

    rating_score : float
        Normalized movie rating (0-1)

    popularity_score : float
        Normalized popularity score (0-1)

    Returns
    -------
    float
        Final weighted recommendation score
    """

    return (
        SIMILARITY_WEIGHT * similarity_score +
        RATING_WEIGHT * rating_score +
        POPULARITY_WEIGHT * popularity_score
    )

In [53]:
def hybrid_recommend(movie_name, top_n=10):

    # -------------------------------------
    # Check Movie Exists
    # -------------------------------------

    if movie_name not in movies["Title"].values:

        print("Movie Not Found")

        return None

    # -------------------------------------
    # Movie Index
    # -------------------------------------

    index = movies[
        movies["Title"] == movie_name
    ].index[0]

    # -------------------------------------
    # Similarity Scores
    # -------------------------------------

    similarity_scores = similarity[index]

    recommendations = []

    # -------------------------------------
    # Calculate Final Score
    # -------------------------------------

    for i, sim_score in enumerate(similarity_scores):

        if i == index:
            continue

        rating = movies.iloc[i]["Rating_Score"]

        popularity = movies.iloc[i]["Popularity_Score"]

        final_score = calculate_hybrid_score(
            sim_score,
            rating,
            popularity
        )

        recommendations.append({

            "Title": movies.iloc[i]["Title"],

            "Genre": movies.iloc[i]["Genre"],

            "Year": movies.iloc[i]["Year"],

            "Rating": movies.iloc[i]["Vote_Average"],

            "Popularity": movies.iloc[i]["Popularity"],

            "Poster": movies.iloc[i]["Poster_Url"],

            "Similarity": round(sim_score,3),

            "HybridScore": round(final_score,3)

        })

    recommendations = pd.DataFrame(recommendations)

    recommendations = recommendations.sort_values(

        by="HybridScore",

        ascending=False

    )

    return recommendations.head(top_n)

In [54]:
hybrid_recommend("No Exit")

,Title,Genre,Year,Rating,Popularity,Poster,Similarity,HybridScore
2578,Kidnap,"drama, thriller",2017,6.2,34.020,https://image.tmdb.org/t/p/original/fwdOX3PEZl...,0.253,0.301
2,Encanto,"animation, comedy, family, fantasy",2021,7.7,2402.201,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,0.136,0.296
3231,Martyrs,"horror, drama, thriller",2008,7.4,28.531,https://image.tmdb.org/t/p/original/do92C3aiAD...,0.196,0.286
8672,Requiem for a Dream,"crime, drama",2000,8.0,14.448,https://image.tmdb.org/t/p/original/a1BrBoErnd...,0.172,0.281
4073,"Girl, Interrupted",drama,1999,7.6,24.154,https://image.tmdb.org/t/p/original/dOBdatHIVp...,0.182,0.280
0,Spider-Man: No Way Home,"action, adventure, science fiction",2021,8.3,5083.954,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,0.021,0.280
4309,A Street Cat Named Bob,"family, drama",2016,7.9,23.232,https://image.tmdb.org/t/p/original/nBYG0D2Fcb...,0.151,0.264
8051,The Tunnel,"thriller, drama",2019,6.2,15.174,https://image.tmdb.org/t/p/original/x2Nkkk0znw...,0.192,0.258
4739,Donnie Brasco,"crime, drama, thriller",1997,7.5,21.763,https://image.tmdb.org/t/p/original/xtKLvpOfAR...,0.148,0.254
2314,[REC],"horror, mystery",2007,7.2,36.562,https://image.tmdb.org/t/p/original/5XsVGgo8I1...,0.155,0.253


In [55]:
hybrid_recommend("No Exit")

,Title,Genre,Year,Rating,Popularity,Poster,Similarity,HybridScore
2578,Kidnap,"drama, thriller",2017,6.2,34.020,https://image.tmdb.org/t/p/original/fwdOX3PEZl...,0.253,0.301
2,Encanto,"animation, comedy, family, fantasy",2021,7.7,2402.201,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,0.136,0.296
3231,Martyrs,"horror, drama, thriller",2008,7.4,28.531,https://image.tmdb.org/t/p/original/do92C3aiAD...,0.196,0.286
8672,Requiem for a Dream,"crime, drama",2000,8.0,14.448,https://image.tmdb.org/t/p/original/a1BrBoErnd...,0.172,0.281
4073,"Girl, Interrupted",drama,1999,7.6,24.154,https://image.tmdb.org/t/p/original/dOBdatHIVp...,0.182,0.280
0,Spider-Man: No Way Home,"action, adventure, science fiction",2021,8.3,5083.954,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,0.021,0.280
4309,A Street Cat Named Bob,"family, drama",2016,7.9,23.232,https://image.tmdb.org/t/p/original/nBYG0D2Fcb...,0.151,0.264
8051,The Tunnel,"thriller, drama",2019,6.2,15.174,https://image.tmdb.org/t/p/original/x2Nkkk0znw...,0.192,0.258
4739,Donnie Brasco,"crime, drama, thriller",1997,7.5,21.763,https://image.tmdb.org/t/p/original/xtKLvpOfAR...,0.148,0.254
2314,[REC],"horror, mystery",2007,7.2,36.562,https://image.tmdb.org/t/p/original/5XsVGgo8I1...,0.155,0.253


In [56]:
top_rated = movies.sort_values(

    by="Vote_Average",

    ascending=False

)

top_rated[
    [
        "Title",
        "Vote_Average"
    ]
].head(10)

,Title,Vote_Average
9390,Kung Fu Master Huo Yuanjia,10.0
7338,Franco Escamilla: Por La Anécdota,9.2
667,Demon Slayer: Kimetsu no Yaiba Sibling's Bond,9.1
2324,Impossible Things,9.1
7013,Sex School: Dorms of Desire,9.0
2390,The Three Deaths of Marisela Escobedo,9.0
7400,My Sex Doll,9.0
6727,Mission «Sky»,9.0
7038,Bring the Soul: The Movie,8.9
8646,Burn the Stage: The Movie,8.9


In [57]:
trending = movies.sort_values(

    by="Popularity",

    ascending=False

)

trending[
    [
        "Title",
        "Popularity"
    ]
].head(10)

,Title,Popularity
0,Spider-Man: No Way Home,5083.954
1,The Batman,3827.658
2,No Exit,2618.087
3,Encanto,2402.201
4,The King's Man,1895.511
5,The Commando,1750.484
6,Scream,1675.161
7,Kimi,1601.782
8,Fistful of Vengeance,1594.013
9,Eternals,1537.406


In [58]:
pickle.dump(

    movies,

    open("../models/movies.pkl","wb")

)

In [59]:
movies = pickle.load(

    open("../models/movies.pkl","rb")

)

movies.head()

,Title,Tags,Genre,Overview,Original_Language,Popularity,Vote_Average,Vote_Count,Poster_Url,Year,Popularity_Score,Rating_Score
0,Spider-Man: No Way Home,action adventur scienc fiction peter parker un...,"action, adventure, science fiction",peter parker is unmasked and no longer able to...,en,5083.954,8.3,8940.0,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,2021,1.000000,0.83
1,The Batman,crime mysteri thriller second year fight crime...,"crime, mystery, thriller","in his second year of fighting crime, batman u...",en,3827.658,8.1,1151.0,https://image.tmdb.org/t/p/original/74xTEgt7R3...,2022,0.752239,0.81
2,No Exit,thriller strand rest stop mountain blizzard re...,thriller,stranded at a rest stop in the mountains durin...,en,2618.087,6.3,122.0,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,2022,0.513693,0.63
3,Encanto,anim comedi famili fantasi tale extraordinari ...,"animation, comedy, family, fantasy","the tale of an extraordinary family, the madri...",en,2402.201,7.7,5076.0,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,2021,0.471117,0.77
4,The King's Man,action adventur thriller war collect histori w...,"action, adventure, thriller, war",as a collection of history's worst tyrants and...,en,1895.511,7.0,1793.0,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,2021,0.371190,0.70


In [60]:
print(movies.columns)

Index(['Title', 'Tags', 'Genre', 'Overview', 'Original_Language', 'Popularity',
       'Vote_Average', 'Vote_Count', 'Poster_Url', 'Year', 'Popularity_Score',
       'Rating_Score'],
      dtype='object')
